# Imports

In [1]:
import sys
sys.path.append("..")

In [2]:
import os
from rouge_score import rouge_scorer
import re
import pymorphy3
import string
import logging
from datasets import load_dataset

In [3]:
def log_module(name, level):
    logger = logging.getLogger(name)
    logger.setLevel(level)

    ch = logging.StreamHandler()
    ch.setLevel(level)

    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    ch.setFormatter(formatter)

    logger.addHandler(ch)

In [4]:
log_module("GraphEvolve", logging.DEBUG)
log_module("AGoTI", logging.WARNING)

In [5]:
from GraphEvolve.task import SimpleTask
from GraphEvolve.goo import get_default_config
from GraphEvolve.planner import ThreeStepPlanner
from GraphEvolve.solution_database import SolutionDatabase
from GraphEvolve.sampler import PowerLawSampler
from GraphEvolve.mutator import EditMutator, PlanMutator, RandomMutator
from GraphEvolve.runner import Runner

In [6]:
from AGoTI.model import LLM, ApiVLLMModel

# Models

In [7]:
generation_config = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 5000,
    "extra_body": {
        "repetition_penalty": 1.0,
        "guided_choice": None,
        "add_generation_prompt": True,
        "guided_regex": None
    }
}

In [8]:
runner_generation_config = {
    "temperature": 0.1,
    "top_p": 0.9,
    "max_tokens": 3000,
    "extra_body": {
        "repetition_penalty": 1.0,
        "guided_choice": None,
        "add_generation_prompt": True,
        "guided_regex": None,
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
}

In [9]:
API_KEY = os.getenv("API_KEY")
API_URL = os.getenv("API_URL")
model_name = "Qwen3-235B-A22B-Instruct-2507"
runner_model_name = "tpro-2"
model = ApiVLLMModel(API_KEY, API_URL, model_name, generation_config=generation_config)
runner_model = ApiVLLMModel(API_KEY, API_URL, runner_model_name, generation_config=generation_config)

# Task

In [ ]:
FEEDBACK_PROMPT = """Оцени качество аннотации текста. Исходный текст:
\"\"\"
{text}
\"\"\"

Предложенная аннтоация:
\"\"\"
{y_pred}
\"\"\"

Эталонная аннотация:
\"\"\"
{y_true}
\"\"\"

Напиши краткий фидбек (1-2 предложения), сравнив предложенную и эталонную аннотации. Подчеркни основные моменты, которые необходимо исправить."""

In [ ]:
TASK_DESCRIPTION = \
"""Твоя задача состоит в том, чтобы написать краткую аннотацию (2-3 предложения) на русском языке к представленному тексту. \
Текст обозначается строкой {text}. Ответ должен быть в пункте \"summary\""""

In [12]:
def mean(arr):
    return sum(arr) / max(len(arr), 1)

class CustomTokenizer():
    def __init__(self):
        self.punkt_regex = re.compile('[%s]' % re.escape(string.punctuation))
        self.morph = pymorphy3.MorphAnalyzer()

    def tokenize(self, text):
        return self.preprocess_sentence(text)

    def preprocess_sentence(self, text: str):
        text = text.lower()
        text = self.punkt_regex.sub(' ', text)
        words = [self.morph.parse(token)[0].normal_form for token in text.split() if len(token.strip()) > 0]
        return words

rouge_scorer = rouge_scorer.RougeScorer(['rougeL'], tokenizer=CustomTokenizer())

def rougel(gold, generated):
    return rouge_scorer.score(gold, generated)['rougeL']

In [ ]:
class Gazeta(SimpleTask):
    def __init__(self, model: LLM):
        self.model = model
        self.description = TASK_DESCRIPTION

    def dataset_args(self):
        return {"path": "IlyaGusev/gazeta"}

    def load_dataset(self):
        dataset = load_dataset(**self.dataset_args())

        train_dataset = dataset[self.train_split()]
        test_dataset = dataset[self.test_split()]

        train_dataset = train_dataset.select(range(min(10, len(train_dataset))))
        test_dataset = test_dataset.select(range(min(100, len(test_dataset))))

        def preprocess(sample):
            return {
                'text': sample['title'].strip() + '\n' + sample['text'].strip(),
                'summary': sample['summary'].strip()
            }
        
        train_dataset = train_dataset.map(
            preprocess,
            remove_columns=['title', 'date', 'url']
        )
        test_dataset = test_dataset.map(
            preprocess,
            remove_columns=['title', 'date', 'url']
        )

        return (train_dataset, test_dataset)

    def train_split(self):
        return "validation"

    def test_split(self):
        return "test"

    async def evaluate(self, sample, output):
        y_pred = output.get("summary", [""])
        if len(y_pred) == 0:
            return 0
        y_pred = y_pred[0]
        y_true = sample['summary']
        score = rougel(y_true, y_pred).fmeasure
        return score

    async def aggregate(self, scores):
        return mean(scores)

    async def generate_feedback(self, sample, output) -> str:
        y_pred = output.get("summary", [""])
        if len(y_pred) == 0:
            return "Ответ пустой, задача не решена!"
        y_pred = y_pred[0]
        y_true = sample["inputs"][self.target_lang]
        prompt = FEEDBACK_PROMPT\
            .replace("{text}", sample["text"])\
            .replace("{y_pred}", y_pred)\
            .replace("{y_true}", y_true)
        response = await model.generate([{"role": "user", "content": prompt}])
        return response

    def get_goo_kwargs(self, sample):
        return {"replace": {"{text}": sample["text"]}}

In [ ]:
task = Gazeta(model_name)

# GoO config

In [15]:
goo_config = get_default_config(runner_model)

# Planner

In [16]:
planner = ThreeStepPlanner(model, goo_config)

# Solution database

In [ ]:
solution_db = SolutionDatabase(goo_config, clear_db=False, db_path="./gz_solution_database.db")

In [ ]:
print("\n".join(f"{score:.3f} (id: {id})" for (score, id), _ in solution_db.solutions._data))

# Sampler

In [18]:
sampler = PowerLawSampler(solution_db)

# Mutator

In [19]:
edit_mutator = EditMutator(model, goo_config)
plan_mutator = PlanMutator(model, goo_config, planner)
mutator = RandomMutator([edit_mutator, plan_mutator], [0.75, 0.25])

# Basic solution

In [ ]:
basic_plan_draft = """1. Написать краткую аннотацию на русском языке к представленному тексту."""

In [ ]:
basic_plan_formal = """\
1. "корень"
"Аннотатор"
"Пишет краткую аннотацию (2-3 предложения) на русском языке к представленному тексту (обозначается строкой {text}). Результат — аннотация."
входы: []
выход: summary

Ответы: {"summary": 1}"""

In [ ]:
basic_goo_json = {'nodes': {'1': {'class': 'корень',
   'parents': [],
   'args': {'prompt': 'Твоя задача состоит в том, чтобы написать краткую аннотацию (2-3 предложения) на русском языке к представленному тексту. Не приводи никаких размышлений или объяснений, только аннотация.\n**Текст:**\n{text}',
    'parsing': 'none',
    'thought_tag': 'summary',
    'name': "Аннотатор",
    'description': "Пишет краткую аннотацию (2-3 предложения) на русском языке к представленному тексту (обозначается строкой {text}). Результат — аннотация."
   }}},
"roots": [1],
"outputs": {"summary": 1}}

# Runner

In [ ]:
runner = Runner(task, solution_db, goo_config, planner, sampler, mutator)

In [ ]:
INIT_WITH_BASIC_SOLUTION = True

In [ ]:
if INIT_WITH_BASIC_SOLUTION:
    basic_solution = await runner.run_solution(basic_plan_draft, basic_plan_formal, basic_goo_json)
    solution_db.add_root(basic_solution)

In [ ]:
await runner.run(1.0, 10)

# Results

In [ ]:
print("\n".join(f"{score:.3f} (id: {id})" for (score, id), _ in solution_db.solutions._data))

## Initial solution

In [ ]:
print(f"Train: {solution_db.root_solution.score}")
root_score = await runner.run_test(solution_db.root_solution)
print(f"Test: {root_score}")

## Best solution

In [ ]:
print(f"Train: {solution_db.best_solution.score}")
best_score = await runner.run_test(solution_db.best_solution)
print(f"Test: {best_score}")

## Basic solution

In [ ]:
if not INIT_WITH_BASIC_SOLUTION:
    basic_solution = await runner.run_solution("", "", basic_goo_json)
    print(f"Train: {basic_solution.score}")

In [ ]:
if not INIT_WITH_BASIC_SOLUTION:
    basic_score = await runner.run_test(basic_solution)
    print(f"Test: {basic_score}")